# Lab 7 — Training a CNN that works: an industrial inspection case

**Deep Learning for Computer Vision — Day 2** · ECAM, 5MAU · 2026–2027

A line produces machined flanges. A camera above the conveyor must sort them into 4 classes:
`ok`, `scratch`, `missing_hole` (a bolt hole was not drilled) and `stain` (oil stain).

**The catch** — typical of real projects: the **training** images were all acquired during *one* session, with the part placed by hand (almost always the same orientation, constant lighting). The **validation** and **test** images come from the production line: any orientation, varying lighting, small offsets.

**Goals.**
1. Diagnose a model from its learning curves.
2. Use **data augmentation** to make the training data look like production.
3. Use **batch normalisation**, a learning-rate schedule, weight decay, and **global average pooling**.
4. Evaluate properly: confusion matrix, precision/recall per class, error analysis — on the **test set, once**.
5. Mini-competition: best validation accuracy within a fixed budget.

**Duration:** ~60 min. Each training run takes ≈ 1 min on a laptop CPU.

> *References:* Stanford CS231n, Lecture 6 (*Training CNNs and CNN architectures*), 2025 · KenN7, *Deep learning for computer vision*, ECAM, cours 3 and the *Mini Waffles* defect-classification lab, https://kenn7.github.io/deep_learning/ · S. Ioffe, C. Szegedy, "Batch normalization", ICML 2015 · L. Smith, N. Topin, "Super-convergence: very fast training of neural networks using large learning rates" (one-cycle), 2019 · M. Lin, Q. Chen, S. Yan, "Network in network" (global average pooling), ICLR 2014. Dataset: synthetic, generated by `tools/make_parts_dataset.py`.

In [ ]:
# Setup check: are all the packages of requirements.txt installed in this kernel?
import importlib
import importlib.util

importlib.invalidate_caches()
REQUIRED = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy", "matplotlib": "matplotlib",
            "cv2": "opencv-contrib-python", "ipywidgets": "ipywidgets"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                     "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
print("Setup OK: all required packages are installed.")

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms as T

DATA = Path("data") if Path("data").exists() else Path("../data")
PARTS = DATA / "parts"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(max(1, torch.get_num_threads()))

## 1. The data

In [ ]:
MEAN, STD = 0.30, 0.25  # statistics of the training images (grey levels in [0, 1])
eval_tfm = T.Compose([T.Grayscale(), T.ToTensor(), T.Normalize([MEAN], [STD])])

train_plain = datasets.ImageFolder(PARTS / "train", eval_tfm)
val_set = datasets.ImageFolder(PARTS / "val", eval_tfm)
test_set = datasets.ImageFolder(PARTS / "test", eval_tfm)
classes = train_plain.classes
print(classes)
print("train", len(train_plain), "| val", len(val_set), "| test", len(test_set))
print("images per class (train):", np.bincount(train_plain.targets))

fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for row, (ds, name) in enumerate([(train_plain, "train"), (val_set, "val")]):
    idx = np.random.default_rng(1).choice(len(ds), 8, replace=False)
    for a, i in zip(ax[row], idx):
        x, y = ds[i]
        a.imshow(x[0] * STD + MEAN, cmap="gray", vmin=0, vmax=1)
        a.set_title(f"{name}: {classes[y]}", fontsize=8); a.axis("off")

**Question.** Look at the two rows. List the differences between the training and the validation images.

## 2. Tools: training with history

`fit` is given. It supports the options used later (`weight_decay`, `one_cycle`). It keeps the weights of the epoch with the best validation accuracy (early stopping by checkpoint).

In [ ]:
def evaluate(model, loader):
    model.eval()
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total += F.cross_entropy(logits, y, reduction="sum").item()
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return total / n, correct / n


def fit(model, train_tfm, epochs=12, lr=1e-3, weight_decay=0.0, one_cycle=False, seed=0, verbose=True):
    torch.manual_seed(seed)
    train_loader = torch.utils.data.DataLoader(datasets.ImageFolder(PARTS / "train", train_tfm),
                                               batch_size=32, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_set, batch_size=256)
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = (torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader))
             if one_cycle else None)
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best, best_state, t0 = 0.0, None, time.perf_counter()
    for epoch in range(epochs):
        model.train()
        tl, tc, tn = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            if sched:
                sched.step()
            tl += loss.item() * len(y); tc += (logits.argmax(1) == y).sum().item(); tn += len(y)
        vl, va = evaluate(model, val_loader)
        for k, v in zip(hist, [tl / tn, tc / tn, vl, va]):
            hist[k].append(v)
        if va > best:
            best, best_state = va, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    if verbose:
        print(f"best val acc {best:.3f} | last train acc {hist['train_acc'][-1]:.3f} "
              f"| {sum(p.numel() for p in model.parameters())} params | {time.perf_counter() - t0:.0f}s")
    return hist


def plot_history(hist, title=""):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3))
    ax[0].plot(hist["train_loss"], "--", label="train"); ax[0].plot(hist["val_loss"], label="val"); ax[0].set_title("loss")
    ax[1].plot(hist["train_acc"], "--", label="train"); ax[1].plot(hist["val_acc"], label="val"); ax[1].set_title("accuracy")
    for a in ax:
        a.set_xlabel("epoch"); a.legend()
    fig.suptitle(title)


results = {}

## 3. Experiment A — baseline

Three blocks *conv–ReLU–pool* and a fully connected classifier, no augmentation.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16 x 32 x 32
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 x 16 x 16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 64 x 8 x 8
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64 * 8 * 8, 128), nn.ReLU(), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


hist = fit(BaselineCNN(), eval_tfm)
results["A baseline"] = max(hist["val_acc"])
plot_history(hist, "A — baseline, no augmentation")

**Question.** Diagnose: under-fitting or over-fitting? What does the gap between train and val tell you here, given section 1?

## 4. Experiment B — data augmentation

Augmentation applies **random, label-preserving** transformations to every training image at every epoch, so the model sees a new variant each time. Choose transformations that are **physically plausible** on the line:
* any rotation (`T.RandomRotation(180)`), small translations (`T.RandomAffine(0, translate=(0.08, 0.08))`),
* brightness changes (`T.ColorJitter(brightness=0.4)`),
* horizontal flips (the parts are symmetric).

Augmentation is **only** for the training set — never for validation/test.

### Exercise 4.1
Build `aug_tfm` (the `Grayscale`, `ToTensor` and `Normalize` must stay, `ToTensor` after the PIL-image transforms) and retrain the baseline.

In [ ]:
# TODO
aug_tfm = T.Compose([...])

ds_aug = datasets.ImageFolder(PARTS / "train", aug_tfm)
fig, ax = plt.subplots(1, 8, figsize=(14, 2))
for a in ax:
    a.imshow(ds_aug[0][0][0] * STD + MEAN, cmap="gray", vmin=0, vmax=1); a.axis("off")
fig.suptitle("8 random augmentations of the same training image")

hist = fit(BaselineCNN(), aug_tfm)
results["B + augmentation"] = max(hist["val_acc"])
plot_history(hist, "B — baseline + augmentation")
assert results["B + augmentation"] > results["A baseline"]

**Question.** The training accuracy is now *lower* than in A. Is it a problem?

## 5. Experiment C — a deeper network with batch normalisation

**Batch normalisation** normalises the activations of each channel over the mini-batch (then rescales them with learned $\gamma, \beta$). It stabilises and speeds up training, allowing deeper networks and larger learning rates. Pattern: `Conv → BatchNorm → ReLU`.

We also use **AdamW** with weight decay and the **one-cycle** learning-rate schedule (warm-up then decay).

### Exercise 5.1
Complete `block(c_in, c_out)`: two `Conv(3×3, padding 1) → BatchNorm2d → ReLU` then `MaxPool2d(2)`. The rest of `DeepCNN` is given (same classifier as the baseline, with dropout).

In [ ]:
def block(c_in, c_out):
    # TODO
    ...


class DeepCNN(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.features = nn.Sequential(block(1, 16), block(16, 32), block(32, 64))  # 64 x 8 x 8
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
                                        nn.Dropout(0.2), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


hist = fit(DeepCNN(), aug_tfm, lr=3e-3, weight_decay=1e-4, one_cycle=True)
results["C + BatchNorm, deeper, one-cycle"] = max(hist["val_acc"])
plot_history(hist, "C — deeper CNN with batch norm")

## 6. Experiment D — global average pooling

The classifier of A–C flattens a `64 × 8 × 8` map: its 4 096 inputs are tied to **positions**. A rotated defect lands at other positions, and this layer alone holds > 500 000 parameters.

**Global average pooling** (GAP) averages each feature map over all positions → a vector of 64 values, followed by a single linear layer. The question becomes "*is there* a scratch somewhere?" instead of "*is there* a scratch at position (3, 5)?".

### Exercise 6.1
Write `GapCNN`: same `features` as `DeepCNN`, classifier `AdaptiveAvgPool2d(1) → Flatten → Linear(64, n_classes)`. Train it with the same settings. Compare the number of parameters.

In [ ]:
class GapCNN(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        # TODO
        ...

    def forward(self, x):
        return self.classifier(self.features(x))


gap_model = GapCNN()
hist = fit(gap_model, aug_tfm, lr=3e-3, weight_decay=1e-4, one_cycle=True)
results["D + global average pooling"] = max(hist["val_acc"])
plot_history(hist, "D — global average pooling")
assert results["D + global average pooling"] > 0.9

In [ ]:
plt.figure(figsize=(9, 3))
plt.barh(list(results), list(results.values()), color="#003366")
for k, v in enumerate(results.values()):
    plt.text(v + 0.01, k, f"{v:.3f}", va="center")
plt.xlim(0, 1.1); plt.xlabel("best validation accuracy"); plt.gca().invert_yaxis();

## 7. Final evaluation on the test set

The validation set was used to *choose* the model (we looked at it many times): its accuracy is optimistic. The **test set** is used **once**, at the end, to report the performance.

### Exercise 7.1
1. Compute the predictions of `gap_model` on the test set and build the **confusion matrix** `cm[true, predicted]` (4×4) with NumPy only.
2. From `cm`, compute the **precision** and **recall** of each class.
3. In this application, which error is the most expensive: a defective part classified `ok`, or a good part classified as defective? Which metric measures it?

In [ ]:
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256)
gap_model.eval()
ys, preds = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.append(gap_model(x.to(device)).argmax(1).cpu())
        ys.append(y)
ys, preds = torch.cat(ys).numpy(), torch.cat(preds).numpy()

# TODO
cm = ... ; precision = ... ; recall = ...

print(f"test accuracy: {np.trace(cm) / cm.sum():.3f}")
for c, p, r in zip(classes, precision, recall):
    print(f"{c:13s} precision {p:.3f}  recall {r:.3f}")
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, v, ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
ax.set_xticks(range(4), classes, rotation=30); ax.set_yticks(range(4), classes)
ax.set_xlabel("predicted"); ax.set_ylabel("true");
assert cm.sum() == len(test_set)

### Exercise 7.2 — Error analysis
Display the misclassified test images with their true and predicted classes. Are the errors understandable? What would you do to fix them?

In [ ]:
wrong = np.nonzero(preds != ys)[0][:12]
fig, ax = plt.subplots(1, max(len(wrong), 1), figsize=(1.6 * max(len(wrong), 1), 2.2), squeeze=False)
for a, i in zip(ax[0], wrong):
    a.imshow(test_set[i][0][0] * STD + MEAN, cmap="gray", vmin=0, vmax=1); a.axis("off")
    a.set_title(f"T: {classes[ys[i]]}\nP: {classes[preds[i]]}", fontsize=7)

In [ ]:
torch.save(gap_model.state_dict(), "parts_cnn.pt")  # reused in Lab 8 (Grad-CAM)
print("saved parts_cnn.pt")

## 8. Mini-competition (if time allows)

Rules: training budget **15 epochs**, `aug_tfm` may be changed, any architecture **< 200 000 parameters**, selection on the validation set only. Report your best validation accuracy and your number of parameters. Ideas: other augmentations (`RandomResizedCrop`, `GaussianBlur`), wider/narrower blocks, dropout before the last layer, other learning rates, label smoothing (`F.cross_entropy(..., label_smoothing=0.1)`).

In [ ]:
# your experiments here